In [89]:
import torch
from torch import nn, Tensor
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.utils.data import Dataset, random_split
from torch.nn.utils.rnn import pad_sequence
import json
import random
import math
from torch.utils.tensorboard import SummaryWriter

#For cutting and increase the Median of the timestamps
THRESHOLD_TIMESTAMPS = 16

#Part of TRACE paper, after to cut if the sequence_lenght of the session is more bigger cut, if less padd
L_SEQUENCE_LENGHT = 64

#The learning embeddings for the Transformer_encoder_layer
EMBEDDING_DIM = 32

EPOCHS = 10

TRAINING_DATA_SET = 0.98
#DEV_SET  = 0.01
TEST_SET = 0.02

In [90]:
def extract_json(filename: str):
    """
    Extracts the json that are used in the project (OttoDataset) "train.jsonl"
    - input -> filename : the name of the session
    - return all the sessions
    """
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

In [91]:
sessions_bf_threshold = []

for i, (session_id, eventstotal) in enumerate(extract_json("train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions_bf_threshold.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 200:
        break

In [92]:
class OttoDataSetSession(Dataset):
    """
    The first Dataset, before the TRACE Paper Cut Threshold of the Timestamps which in this case is 16 (THRESHOLD_TIMESTAMPS) 
    It Transforms the click aids in numbers 
    """
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)

    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [93]:
sessions_bf_threshold = OttoDataSetSession(sessions_bf_threshold)
print(f"Total before threshold len of the Sessions: {len(sessions_bf_threshold)}")

#Session After the Threshold 
session_sample_lenght_after_threshold = []

for i in range(len(sessions_bf_threshold)):
    sample = sessions_bf_threshold[i]
    if len(sample["timestamps"]) >= THRESHOLD_TIMESTAMPS:
        session_sample_lenght_after_threshold.append(sample)
        
print(f"Total After threshold len of the Sessions: {len(session_sample_lenght_after_threshold)}")

Total before threshold len of the Sessions: 201
Total After threshold len of the Sessions: 154


In [94]:

def most_frequent(List):
    """
    Counts the highest number of timestamps per product, used in Logit SAT4 (Seeing the same Aid 4 times)
    """
    dict = {}
    count, itm = 0, ''
    for item in reversed(List):
        dict[item] = dict.get(item, 0) + 1
        if dict[item] >= count :
            count, itm = dict[item], item
    return(count, itm)


def most_frequent(List):
    """
    Counts the highest number of timestamps per product, used in Logit SAT4 (Seeing the same Aid 4 times)
    """
    dict = {}
    count, itm = 0, ''
    for item in reversed(List):
        dict[item] = dict.get(item, 0) + 1
        if dict[item] >= count :
            count, itm = dict[item], item
    return(count, itm)

def custom_collate(batch):
    inputs = [sample["inputs"] for sample in batch]
    targets = [sample["targets"] for sample in batch]
    
    
    aids = [torch.tensor(d["aid"], dtype=torch.long) for d in inputs]
    timestamps = [torch.tensor(d["timestamps"], dtype=torch.long) for d in inputs]
    events_type = [torch.tensor(d["type"], dtype=torch.int32) for d in inputs]
    
    
    
    aids_padded_input = pad_sequence(aids, batch_first=True, padding_value=0)
    timestamps_padded_input = pad_sequence(timestamps, batch_first=True, padding_value=0)
    events_type_padded_input = pad_sequence(events_type, batch_first=True, padding_value=0)
   
    session_id_inputs = [torch.tensor(d["session_id"], dtype=torch.int64) for d in inputs]
    
    return {
        "inputs" :{
        "aid" : aids_padded_input,
        "timestamps" : timestamps_padded_input,
        "type" : events_type_padded_input,
        "session_id" : session_id_inputs    
        },
        "target" : targets
        }
    
def replace_zeros_with_prev_nonzero(tensor):
    output = tensor.clone()
    for i in range(len(output)):
        prev_value = 0
        for j in range(len(tensor[i])):
            if tensor[i,j] == 0:
                output[i,j] = prev_value
            else:
                prev_value = tensor[i,j].item()      
    return output


In [95]:
class TraceOttoDataSet(OttoDataSetSession):
    def __init__(self, session):
        super().__init__(session)
        
        #Cut: input/target
        self.inputs_part, self.target_part = self.__cut_input_target__()
        
        self.input = self.inputs_part
        
        self.train_input = self.__sequenceLenght__()
        self.target = self.target_part
         
        self.ATC = self.__ATC_task_logit__()
        self.SAT = self.__SAT__task_logit__()
        self.PD1 = self.__PD1_task_logit___()
        self.RA1 = self.__RA1_task_logit___()
    
    def __getitem__(self, index):
        inputs_sessions = {
            "session_id": self.train_input[index]["session_id"],
            "aid": self.train_input[index]["aid"],
            "timestamps": self.train_input[index]["timestamps"],
            "type": self.train_input[index]["type"],
        }

        #4 classes
        target_sessions = {
            "ATC": self.ATC[index],
            "SAT": self.SAT[index],
            "PD1": self.PD1[index],
            "RA1": self.RA1[index] ,
            }

        return {
            "inputs": inputs_sessions,
            "targets": target_sessions
        }
     
        
    
    def __padding__(self, session):   
        padd_len = L_SEQUENCE_LENGHT - len(session["timestamps"])
        zeros = torch.zeros(padd_len, dtype=session["aid"].dtype)
        
        aid_padded = torch.concat((session["aid"], zeros))
        timestamps_padded = torch.concat((session["timestamps"], zeros))
        type_padded = torch.concat((session["type"], zeros))
        return {
            "session_id":session["session_id"],
            "aid": aid_padded,
            "timestamps": timestamps_padded,
            "type": type_padded
        }
           
        
    def __cut_input_target__(self, min_value=0.80, max_value=0.90):
        input_batches = []
        target_batches = []

        for single_session in self.session:
            cutting_number = random.uniform(min_value, max_value)

            
            n_events = len(single_session["timestamps"])
            input_size = int(n_events * cutting_number)

            input_part = {
                "session_id": single_session["session_id"],
                "aid": single_session["aid"][:input_size],
                "timestamps": single_session["timestamps"][:input_size],
                "type": single_session["type"][:input_size]
            }

            target_part = {
                "session_id": single_session["session_id"],
                "aid": single_session["aid"][input_size:],
                "timestamps": single_session["timestamps"][input_size:],
                "type": single_session["type"][input_size:]
            }

            input_batches.append(input_part)
            target_batches.append(target_part)

        return input_batches, target_batches


            
    def __sequenceLenght__(self):
        sessions_after_sequence_lenght = []
        for session in self.input:
            if len(session["timestamps"]) >= L_SEQUENCE_LENGHT:
                sessions_after_sequence_lenght.append({
                "session_id": session["session_id"],
                "aid": session["aid"][:L_SEQUENCE_LENGHT],
                "timestamps": session["timestamps"][:L_SEQUENCE_LENGHT],
                "type": session["type"][:L_SEQUENCE_LENGHT]
            })
            else: 
                sessions_after_sequence_lenght.append(self.__padding__(session))
        return sessions_after_sequence_lenght
    
    
    """
    Logits of the the model
    """
    def __ATC_task_logit__(self):
        """
        Labels for ATC (User added to the cart in the FUTURE)
        """
        logits_ATC = []
        for target_part in self.target:
            if 3 in target_part["type"]:
                logits_ATC.append(1)
            else:
                logits_ATC.append(0)

        return logits_ATC
    
    def __SAT__task_logit__(self):
        """
        Counts the highest number of timestamps per product, used in Logit SAT4 (Seeing the same Aid 4 times)
        """
        logits_SAT = []
        for session in self.target:
            AidsS_repeated = []
            count = 0    
            for aids in session["aid"]:
                AidsS_repeated.append(aids.item())
                count, product = most_frequent(AidsS_repeated)
            if count >= 4:
                logits_SAT.append(1)
            else:
                logits_SAT.append(0)
        return logits_SAT
    
    def __PD1_task_logit___(self):
        """
        Logits for PD1(Make any Purchase within a day)
        """
        logits_PD1 = []
        ONE_DAY = (86400 * 1000)
        for session in self.target:
            last_ts = session["timestamps"][-1].item()
            ordered_ts = session["timestamps"][session["type"] == 3]
            #Convert into int
            orders_ts = [ts.item() for ts in ordered_ts]
            
            is_purchase = any([(order <= last_ts + ONE_DAY) for order in orders_ts] )
            if is_purchase == True:
                logits_PD1.append(1)
            else:
                logits_PD1.append(0)
            
        return logits_PD1
    
    def __RA1_task_logit___(self):
        """
        Logits for RA1(Return to the same Aid in 1 days)
        """
        ONE_DAY = (86400 * 1000) 
        logits_RA1 = []
        for session in self.target:
            first_aid = session["aid"][0].item()
            first_ts = session["timestamps"][0].item()
            indices = [index for index, aid in enumerate(session["aid"]) if aid.item() == first_aid]
            other_ts_list = session["timestamps"][indices[1:]]

            is_aids = any((other_ts - first_ts) <= ONE_DAY for other_ts in other_ts_list)
            
            if is_aids == True:
                logits_RA1.append(1)
            else: 
                logits_RA1.append(0)
        return logits_RA1
            
           


In [96]:
#DataSet    
Dataset_processed = TraceOttoDataSet(session_sample_lenght_after_threshold)

      
#See if the Lenght of the new inputs are the Lenght Sequence
for i,sample in enumerate(Dataset_processed):
    len_sample = sample["inputs"]
    print(len(len_sample["timestamps"]))
    if i == 0:
        break    


print("================================================ (Logits part) ===================================================")
print("Logits Data_set_processed the ATC (Add to the Cart)")
print(Dataset_processed.__ATC_task_logit__())
    
print("Logits for SAT4 (Seeing the same Aid 4 times)")
print(Dataset_processed.__SAT__task_logit__())
    
print("Logits for PD1 (Make any Purchase within a day)")
print(Dataset_processed.__PD1_task_logit___())
    
print("Logits for RA1 (Return to the same Aid in 1 days)")
print(Dataset_processed.__RA1_task_logit___())

64
================================================ (Logits part) ===================================================
Logits Data_set_processed the ATC (Add to the Cart)
[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
Logits for SAT4 (Seeing the same Aid 4 times)
[0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1,

In [97]:
class MLP(nn.Module):
    def __init__(self, input_channels : int, output_channels : int):
        super().__init__()
        self.layers = nn.Sequential(
                nn.Linear(input_channels, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, output_channels)
            )
    def forward(self, x):
            return self.layers(x)
         
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.d_model = d_model

        
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)  

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        # x: (B, L, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x

In [98]:

class TRACE(nn.Module):
    def __init__(self, num_embeddings_aid : int, 
                 num_embeddings_event_type : int,
                 num_classes : int = 4
               ):
        
        super(TRACE, self).__init__()
        
        self.D_model = EMBEDDING_DIM * 2 + 2 # 66
        
        
        self.embedding_aid = nn.Embedding(num_embeddings=num_embeddings_aid,
                                          embedding_dim=EMBEDDING_DIM)
        
        self.embedding_eventtype = nn.Embedding(num_embeddings=num_embeddings_event_type,
                                                embedding_dim=EMBEDDING_DIM) 

        self.positional_embedding = PositionalEncoding(d_model=self.D_model)
        
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=self.D_model, 
                                                        nhead=6, # 8
                                                        dim_feedforward=128,
                                                        dropout=0.2,
                                                        activation="relu",
                                                        batch_first=True)
        
        self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer,
                                             num_layers=1)
        
                
        self.GBAP = nn.AdaptiveMaxPool1d(output_size=1)
        
        self.__ATC__ = MLP(input_channels=self.D_model, output_channels=1)
        
        """
        self.MLP_layer = MLP(input_channels=self.D_model, 
                             output_channels=num_classes)
        """
        
        
    def forward(self, aids_ids: Tensor, type_ids: Tensor, delta_elapsed : Tensor, delta_between : Tensor) -> Tensor:
        
        #Categorical Learning Embeddings: 32 + 32
        aid_emb = self.embedding_aid(aids_ids)
        type_emb = self.embedding_eventtype(type_ids)
        
        B, L, _ = aid_emb.shape #(Batch, L_seq , 1)
        
        
        #Time Learning Embeddings: 1 + 1
        delta_between = delta_between.unsqueeze(-1) # (Batch, L_seq, 1)


        delta_between = torch.cat([delta_between, delta_between[:, -1:, :]],dim=1)   #(Batch, L_Seq - 1, 1)  
                    
        delta_elapsed = delta_elapsed.unsqueeze(-1).unsqueeze(-1) #(B,) --> #(B, 1, 1)
        
        
        delta_elapsed = delta_elapsed.expand(-1, L, -1) #(B, L_Seq, 1)
          

        x = torch.cat(
            [aid_emb,
            type_emb,
            delta_between,
            delta_elapsed],
            dim=-1)  
            
        
        positional_embed = self.positional_embedding(x)
        
        encoder = self.encoder(positional_embed)
        
        global_avarage_pooling = self.GBAP(encoder.transpose(1,2)).squeeze(-1)
              
        logits = self.__ATC__(global_avarage_pooling)
        
        return logits  

In [99]:

#DataLoaders

train_data, test_data = random_split(dataset=Dataset_processed, lengths=[TRAINING_DATA_SET,TEST_SET])


#DEV_SET
"""
validation_loader = DataLoader(
    dataset=val_data,
    batch_size=32,
    collate_fn=custom_collate,
    shuffle=False
)
"""

#TRAIN SET
train_loader = DataLoader(
    dataset=train_data,
    batch_size=32,
    shuffle=True
)
#TEST SET
test_loader = DataLoader(
    dataset=test_data,
    batch_size=32,
    shuffle=False
)


In [100]:

for batch_training in train_loader:
    sample = batch_training["inputs"]

    print(
        f"Shape Aids: {sample['aid'].shape}, "
        f"Shape Timestamps: {sample['timestamps'].shape}, "
        f"Shape Type: {sample['type'].shape}"
    )
    break

  

Shape Aids: torch.Size([32, 64]), Shape Timestamps: torch.Size([32, 64]), Shape Type: torch.Size([32, 64])


In [101]:
max_aid = max(
    session["inputs"]["aid"].max().item()
    for session in Dataset_processed
)
max_type = max(
    session["inputs"]["type"].max().item()
    for session in Dataset_processed
)


In [ ]:

num_embeddings_aid = max_aid + 1  
num_embeddings_event_type = max_type + 1
trace_model = TRACE(
    num_embeddings_aid=num_embeddings_aid,
    num_embeddings_event_type=num_embeddings_event_type,
    num_classes=4
)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(trace_model.parameters(), lr=1e-5, weight_decay=1e-6)


for epoch in range(EPOCHS):
    trace_model.train()
    epoch_loss = 0.0
    
    for batch_trainer in train_loader:
        evidence = batch_trainer["inputs"]
        
        label_samplel = batch_trainer["targets"]
        
        label = label_samplel["ATC"].unsqueeze_(1)
        
            
        aids = evidence["aid"]
        events_type = evidence["type"]
        timestamps = evidence["timestamps"]      

        zero_mask = timestamps != 0 
        
        last_indices = zero_mask.sum(dim=1) - 1 
        
        ts_first = timestamps[:, 0]
        
        ts_last = timestamps[torch.arange(timestamps.size(0)), last_indices]


        log_delta_elapsed = torch.log1p(torch.clamp(ts_last - ts_first, min=0))

        
        delta_elapsed_normalized = (log_delta_elapsed - log_delta_elapsed.mean()) / (log_delta_elapsed.std() + 1e-6)           # (B,)
    
        #Delta Between, after - before
        delta_between = timestamps[:, 1:] - timestamps[:, :-1]
        
        mask_zero = delta_between > 0 #True if is bigger than 0
        
        
        log_delta_between = torch.log1p(delta_between.clamp(min=0))
        
        mean = (log_delta_between * mask_zero).sum(1, keepdim=True) / mask_zero.sum(1, keepdim=True).clamp(min=1)
        std  = torch.sqrt(((log_delta_between - mean) ** 2 * mask_zero).sum(1, keepdim=True) / mask_zero.sum(1, keepdim=True).clamp(min=1)+ 1e-6 )

        delta_norm_between = torch.where(mask_zero, (log_delta_between - mean) / std, torch.zeros_like(log_delta_between)) #Where 0 is bigger
        print(delta_norm_between)
        
    
        pred = trace_model(
            aids,
            events_type,
            delta_elapsed_normalized,
            delta_norm_between
        )

        loss = criterion(pred, label.float())
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    train_loss = epoch_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {epoch_loss/len(train_loader):.4f}")
    
    """trace_model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_test in test_loader:
            batch_X_test = batch_test["inputs"]
            batch_Y_test = batch_test["targets"]
            batch_ATC_test = batch_Y_test["ATC"]
            
            outputs = trace_model(batch_X_test)
            loss_validation = criterion(outputs, batch_ATC_test)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_ATC_test.cpu().numpy())
    
    val_loss /= len(test_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] Val Loss: {val_loss:.2f}")

            
            """


tensor([[-0.4864, -0.6116,  3.3644,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0273,  0.3212, -0.3853,  ...,  0.0167, -0.0930, -0.1531],
        [-0.8578,  3.4477,  2.5412,  ...,  1.2956, -0.2538, -0.6591],
        ...,
        [ 2.8198,  0.2179, -0.1107,  ...,  2.6041,  2.8790, -0.0124],
        [ 0.8845, -0.3388,  0.4874,  ...,  0.0000,  0.0000,  0.0000],
        [ 2.0263, -0.0807, -0.7466,  ..., -0.4965, -0.6114, -0.7393]])
tensor([[ 1.8132,  0.0721,  0.5257,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.5408,  0.6308, -0.1015,  ...,  3.6865,  0.4442, -0.0878],
        [-0.9128, -0.4360, -0.7862,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [-1.0245, -0.5465, -0.8557,  ..., -0.3771, -0.4014, -0.5252],
        [-0.8604,  2.2173, -0.7818,  ...,  2.0233,  0.2687,  1.6600],
        [-0.6213, -0.2203, -0.6925,  ...,  1.9548, -0.6504, -0.9392]])
tensor([[-1.2180,  0.1797,  0.7798,  ...,  0.8730,  0.4846, -0.0834],
        [-0.7227, -0.7100, -0.5156,  ..., -0.7325,  2.5517,  1

KeyboardInterrupt: 